In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Load team-tournament features and official World Cup final standings.
df = pd.read_csv("../data/processed/clean_data.csv")
cups = pd.read_csv("../data/raw/WorldCups.csv")

print("Clean data shape:", df.shape)
print("World Cups shape:", cups.shape)
df.head()

In [ ]:
# Standardize historical team names across both datasets.
team_name_mapping = {
    "Germany FR": "Germany",
    "USA": "United States",
    "Soviet Union": "Russia",
    'rn">Germany': "Germany",
    'rn">Republic of Ireland': "Republic of Ireland",
    'rn">Serbia and Montenegro': "Serbia and Montenegro",
    'rn">Trinidad and Tobago': "Trinidad and Tobago"
}

# Apply the mapping to the team-level dataset.
df["team"] = df["team"].replace(team_name_mapping)

# Apply the mapping to the semifinal result columns.
semifinal_columns = ["Winner", "Runners-Up", "Third", "Fourth"]
for column in semifinal_columns:
    cups[column] = cups[column].replace(team_name_mapping)

cups[semifinal_columns].head()

In [ ]:
# Convert WorldCups.csv from wide format to one row per semifinalist.
semifinalists = cups.melt(
    id_vars="Year",
    value_vars=semifinal_columns,
    value_name="team"
)[["Year", "team"]].drop_duplicates()

semifinalists["reached_semifinals"] = 1

# Match each team-year row with the official semifinalist list.
df = df.merge(
    semifinalists,
    on=["Year", "team"],
    how="left"
)

df["reached_semifinals"] = df["reached_semifinals"].fillna(0).astype(int)

print(df["reached_semifinals"].value_counts())
print(df["reached_semifinals"].value_counts(normalize=True).round(3))

In [ ]:
# Basic historical confederation mapping used for this beginner model.
confederation_mapping = {
    "Argentina": "CONMEBOL", "Brazil": "CONMEBOL", "Uruguay": "CONMEBOL",
    "Chile": "CONMEBOL", "Colombia": "CONMEBOL", "Paraguay": "CONMEBOL",
    "Peru": "CONMEBOL", "Ecuador": "CONMEBOL", "Bolivia": "CONMEBOL",
    "Germany": "UEFA", "France": "UEFA", "Spain": "UEFA", "Italy": "UEFA",
    "England": "UEFA", "Netherlands": "UEFA", "Portugal": "UEFA",
    "Belgium": "UEFA", "Croatia": "UEFA", "Switzerland": "UEFA",
    "Sweden": "UEFA", "Denmark": "UEFA", "Poland": "UEFA", "Russia": "UEFA",
    "Mexico": "CONCACAF", "United States": "CONCACAF", "Costa Rica": "CONCACAF",
    "Canada": "CONCACAF", "Japan": "AFC", "Korea Republic": "AFC",
    "Iran": "AFC", "Saudi Arabia": "AFC", "Australia": "AFC",
    "Cameroon": "CAF", "Nigeria": "CAF", "Ghana": "CAF",
    "Morocco": "CAF", "Senegal": "CAF", "South Africa": "CAF",
    "New Zealand": "OFC"
}

# Unknown historical teams are grouped as Other to avoid losing rows.
df["confederation"] = df["team"].map(confederation_mapping).fillna("Other")

df[["team", "confederation"]].drop_duplicates().head()

In [ ]:
numeric_features = [
    "matches_played",
    "goals_for",
    "goals_against",
    "wins",
    "draws",
    "losses",
    "average_attendance",
    "goals_per_match",
    "win_percentage",
    "goal_difference"
]

categorical_features = ["confederation"]
target = "reached_semifinals"

model_data = df[["Year", "team"] + numeric_features + categorical_features + [target]].dropna().copy()

print("Model data shape:", model_data.shape)
model_data.head()

In [ ]:
# Convert categorical variables into binary columns.
features = pd.get_dummies(
    model_data[numeric_features + categorical_features],
    columns=categorical_features,
    drop_first=False
)

features[target] = model_data[target].values
features.insert(0, "team", model_data["team"].values)
features.insert(0, "Year", model_data["Year"].values)

features.to_csv("../data/processed/features.csv", index=False)

print("Saved file: ../data/processed/features.csv")
features.head()

In [ ]:
X = features.drop(columns=["Year", "team", target])
y = features[target]

# Scale all model input columns after one-hot encoding.
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Total rows:", X_train.shape[0] + X_test.shape[0])

In [ ]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

joblib.dump(scaler, "../outputs/scaler.pkl")

print("Saved files:")
print("../data/processed/X_train.csv")
print("../data/processed/X_test.csv")
print("../data/processed/y_train.csv")
print("../data/processed/y_test.csv")
print("../outputs/scaler.pkl")